# Loan Default Risk Prediction Using Machine Learning
**Author:** Pravalika Katam  
**Dataset:** Kaggle Loan Risk Analysis — 148,670 rows, 34 columns  
**Target:** `Status` (0 = Non-Default, 1 = Default)

## Step 1 — Data Loading and Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             accuracy_score, precision_score, recall_score)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('Loan_Default.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Count': missing[missing>0], 'Pct': missing_pct[missing>0]}).sort_values('Pct', ascending=False))
print('\nDuplicates:', df.duplicated().sum())
df.describe()

In [ ]:
# Confirm target variable
print('Status value counts:')
print(df['Status'].value_counts())
print('\nMissing rate_of_interest by Status (leakage check):')
print(df.groupby('Status')['rate_of_interest'].apply(lambda x: x.isnull().sum()))
print('\ncredit_type EQUI Status distribution:')
print(df[df['credit_type']=='EQUI']['Status'].value_counts())

### Leakage Investigation
- `rate_of_interest`: Missing for **all 36,439 default rows** and zero non-default rows. Imputing the median creates a near-perfect artificial separator — **dropped**.
- `Interest_rate_spread`: Derived from `rate_of_interest` — **dropped**.
- `Upfront_charges`: Set after loan risk assessment — **dropped**.
- `credit_type = EQUI`: 15,297 of 15,298 rows are defaults (99.99%) — structural anomaly, **dropped**.

## Step 2 — Data Cleaning

In [ ]:
LEAKY = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'credit_type']

df_clean = df.copy()
df_clean.drop(columns=['ID', 'year'] + LEAKY, inplace=True)
print('Shape after dropping leaky + identifier columns:', df_clean.shape)

# Cap impossible LTV values (>200 is physically unrealistic)
ltv_capped = (df_clean['LTV'] > 200).sum()
df_clean.loc[df_clean['LTV'] > 200, 'LTV'] = np.nan
print(f'LTV values capped (>200): {ltv_capped}')

# Median imputation for numeric columns
num_imp = ['term', 'property_value', 'income', 'LTV', 'dtir1']
for col in num_imp:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Mode imputation for categorical columns
cat_imp = ['loan_limit', 'approv_in_adv', 'submission_of_application',
           'age', 'loan_purpose', 'Neg_ammortization']
for col in cat_imp:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('Remaining NaN:', df_clean.isnull().sum().sum())
print('Final clean shape:', df_clean.shape)

## Step 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = df_clean['Status'].value_counts().sort_index()
labels = ['Non-Default (0)', 'Default (1)']
colors = ['#4C72B0', '#DD8452']
axes[0].bar(labels, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+500, f'{v:,}\n({v/len(df_clean)*100:.1f}%)', ha='center', fontsize=9)
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Balance')
plt.tight_layout()
plt.show()
print('Class ratio (0:1):', round(counts[0]/counts[1], 2), ': 1')

In [ ]:
num_features = ['loan_amount', 'property_value', 'income', 'Credit_Score', 'LTV', 'dtir1']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(num_features):
    axes[i].hist(df_clean[col].dropna(), bins=50, color='#4C72B0', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col, fontweight='bold', fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)
plt.suptitle('Numeric Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_features = ['Gender', 'loan_type', 'Region', 'age', 'loan_purpose', 'occupancy_type']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(cat_features):
    vc = df_clean[col].value_counts()
    axes[i].barh(vc.index.astype(str), vc.values, color='#4C72B0', edgecolor='white')
    axes[i].set_title(col, fontweight='bold', fontsize=9)
plt.suptitle('Categorical Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_vs_target = ['loan_type', 'Region', 'Gender', 'age', 'loan_purpose']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
overall_avg = df_clean['Status'].mean() * 100
for i, col in enumerate(cat_vs_target):
    dr = (df_clean.groupby(col)['Status'].mean() * 100).sort_values(ascending=False)
    bar_colors = ['#DD8452' if v > overall_avg else '#4C72B0' for v in dr.values]
    axes[i].bar(dr.index.astype(str), dr.values, color=bar_colors, edgecolor='white')
    axes[i].axhline(y=overall_avg, color='red', linestyle='--', linewidth=1, label='Avg')
    axes[i].set_title(f'Default Rate by {col}', fontweight='bold', fontsize=9)
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=7)
    for j, v in enumerate(dr.values):
        axes[i].text(j, v+0.3, f'{v:.1f}%', ha='center', fontsize=7)
axes[5].set_visible(False)
plt.suptitle('Default Rate by Category', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
num_corr_cols = ['loan_amount', 'term', 'property_value', 'income',
                 'Credit_Score', 'LTV', 'dtir1', 'Status']
corr = df_clean[num_corr_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9}, ax=ax)
ax.set_title('Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
box_cols = ['loan_amount', 'Credit_Score', 'income', 'LTV', 'dtir1']
fig, axes = plt.subplots(1, 5, figsize=(16, 5))
for i, col in enumerate(box_cols):
    df_clean.boxplot(column=col, by='Status', ax=axes[i],
                     medianprops=dict(color='#DD8452', linewidth=2),
                     flierprops=dict(marker='o', markersize=1, alpha=0.2))
    axes[i].set_title(col, fontweight='bold', fontsize=8)
    axes[i].set_xlabel('Status', fontsize=8)
plt.suptitle('Numeric Features vs Target', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing and Feature Engineering

In [ ]:
# Ordinal encoding for age (ordered category)
age_order = {'<25':0, '25-34':1, '35-44':2, '45-54':3, '55-64':4, '65-74':5, '>74':6}
df_clean['age_encoded'] = df_clean['age'].map(age_order)
df_clean.drop(columns=['age'], inplace=True)

# One-hot encode all remaining object columns
obj_cols = df_clean.select_dtypes(include='object').columns.tolist()
print('Columns to one-hot encode:', obj_cols)
df_enc = pd.get_dummies(df_clean, columns=obj_cols, drop_first=True)
print('Shape after encoding:', df_enc.shape)

In [ ]:
X = df_enc.drop(columns=['Status'])
y = df_enc['Status']
print(f'Features: {X.shape[1]}  |  Rows: {X.shape[0]}')
print(f'Class distribution in y: {y.value_counts().to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## Step 5 — Model Training and Evaluation

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=500, class_weight='balanced', random_state=42), True),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42), False),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1), False),
    'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42), False),
}

results = {}
for name, (model, scaled) in models.items():
    Xtr = X_train_sc if scaled else X_train.values
    Xte = X_test_sc  if scaled else X_test.values
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    }
    print(f'\n--- {name} ---')
    print(results[name])
    print(classification_report(y_test, y_pred, target_names=['Non-Default','Default']))

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)

fig, ax = plt.subplots(figsize=(11, 5))
results_df[['Accuracy','Precision','Recall','F1','ROC-AUC']].plot(
    kind='bar', ax=ax, edgecolor='white')
ax.set_title('Model Comparison', fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, (model, scaled) in models.items():
    Xte = X_test_sc if scaled else X_test.values
    y_prob = model.predict_proba(Xte)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = results[name]['ROC-AUC']
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model (Gradient Boosting)
best_name = max(results, key=lambda k: results[k]['ROC-AUC'])
print(f'Best model: {best_name}')
best_model, scaled = models[best_name]
Xte = X_test_sc if scaled else X_test.values
y_pred_best = best_model.predict(Xte)
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Default','Default'],
            yticklabels=['Non-Default','Default'])
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest'][0]
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
top20 = feat_imp.nlargest(20)
fig, ax = plt.subplots(figsize=(9, 7))
top20.sort_values().plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Summary

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.7069 | 0.4363 | 0.6478 | 0.5214 | 0.7480 |
| Decision Tree | 0.8593 | 0.7377 | 0.6659 | 0.7000 | 0.8578 |
| Random Forest | 0.8740 | 0.7823 | 0.6773 | 0.7260 | 0.8808 |
| **Gradient Boosting** | **0.8953** | **0.9364** | **0.6169** | **0.7438** | **0.8925** |

**Best model:** Gradient Boosting (ROC-AUC = 0.8925, F1 = 0.7438)

**Key findings:**
- LTV, property_value, and dtir1 are the most predictive features
- Three columns were removed due to data leakage: `rate_of_interest`, `Interest_rate_spread`, `Upfront_charges`
- `credit_type` was dropped due to the EQUI category structural anomaly (99.99% default rate)
- Target is moderately imbalanced (75/25); `class_weight='balanced'` was used throughout